# UPAR 2027 Track 2 — Clean Colab Runner

This notebook contains **no embedded payload**.

You will:
1. upload the supplied `upar_track2_codabench_aligned_v2.zip`,
2. download the official organizer repository/data,
3. train all models,
4. validate using **mADM**,
5. select the best model,
6. final-train on organizer train+validation,
7. build the CodaBench submission ZIP.


## 0. Enable a GPU runtime

In [1]:
!nvidia-smi || true
!python --version


Sun Oct 18 11:31:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Install versions aligned with CodaBench

In [2]:
!pip -q install -U uv
!uv python install 3.12
!rm -rf /content/uparenv
!uv venv --python 3.12 /content/uparenv

PY = "/content/uparenv/bin/python"

!uv pip install --python {PY} \
  torch==2.4.1 torchvision==0.19.1 \
  --index-url https://download.pytorch.org/whl/cu121

!uv pip install --python {PY} \
  numpy pandas scikit-learn Pillow==10.4.0 tqdm matplotlib gdown

print("Isolated interpreter:", PY)

!{PY} -c "import sys, torch, torchvision; \
print(sys.version); \
print('torch=', torch.__version__); \
print('torchvision=', torchvision.__version__); \
print('CUDA available=', torch.cuda.is_available()); \
print('CUDA runtime=', torch.version.cuda)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 53.4 MB/s eta 0:00:00
Installed Python 3.12.15 in 1.13s
 + cpython-3.12.15-linux-x86_64-gnu (python3.12)
Using CPython 3.12.15
Creating virtual environment at: uparenv
Activate with: source uparenv/bin/activate
Using Python 3.12.15 environment at: uparenv
Resolved 26 packages in 705ms
Prepared 26 packages in 43.98s
Installed 26 packages in 268ms
 + filelock==3.32.3
 + fsspec==2026.7.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.6.1
 + numpy==2.5.2
 + nvidia-cublas-cu12==12.1.3.1
 + nvidia-cuda-cupti-cu12==12.1.105
 + nvidia-cuda-nvrtc-cu12==12.1.105
 + nvidia-cuda-runtime-cu12==12.1.105
 + nvidia-cudnn-cu12==9.1.0.70
 + nvidia-cufft-cu12==11.0.2.54
 + nvidia-curand-cu12==10.3.2.106
 + nvidia-cusolver-cu12==11.4.5.107
 + nvidia-cusparse-cu12==12.1.0.106
 + nvidia-nccl-cu12==2.20.5
 + nvidia-nvjitlink-cu12==12.9.86
 + nvidia-nvtx-cu12==12.1.105
 + pillow==12.3.0
 + setuptools==78.1.0
 + sympy==1.14.0
 + to

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Upload the pipeline ZIP

In [4]:
from google.colab import files
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.endswith(".zip"))
print("Uploaded:", zip_name)


Saving UPAR_Track2_Colab_Bundle.zip to UPAR_Track2_Colab_Bundle.zip
Uploaded: UPAR_Track2_Colab_Bundle.zip


In [5]:
import os, zipfile, shutil
PIPE = "/content/upar_pipeline"
shutil.rmtree(PIPE, ignore_errors=True)
os.makedirs(PIPE, exist_ok=True)

with zipfile.ZipFile(zip_name) as z:
    z.extractall(PIPE)

os.environ["PYTHONPATH"] = PIPE + "/src"
print("Pipeline extracted to:", PIPE)
print(os.listdir(PIPE))


Pipeline extracted to: /content/upar_pipeline
['UPAR_Track2_CLEAN_Colab_Runner.ipynb', 'UPAR_Track2_BUNDLE_README.txt', 'upar_track2_codabench_aligned_v2.zip']


## 3. Download the official UPAR 2027 data

In [6]:
%cd /content
!rm -rf UPAR-Challenge-2027
!git clone https://github.com/speckean/UPAR-Challenge-2027.git
%cd /content/UPAR-Challenge-2027
!python download_datasets.py
PROJECT = "/content/UPAR-Challenge-2027"


/content
Cloning into 'UPAR-Challenge-2027'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 35 (delta 1), reused 0 (delta 0), pack-reused 32 (from 1)
Receiving objects: 100% (35/35), 1.77 MiB | 10.59 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/UPAR-Challenge-2027
Download Market 1501 dataset
Downloading...
From (original): https://drive.google.com/uc?id=0B8-rUzbwVRk0c054eEozWG9COHM
From (redirected): https://drive.google.com/uc?id=0B8-rUzbwVRk0c054eEozWG9COHM&confirm=t&uuid=e6d50fa0-5602-4f51-9502-a4f663e1063b
To: /content/UPAR-Challenge-2027/data/market_1501.zip
100% 153M/153M [00:00<00:00, 214MB/s]
Extract Market 1501 dataset
Extracting : 100% 68042/68042 [00:07<00:00, 9009.76it/s]
Download PA100k dataset
Retrieving folder contents
Processing file 1dt5nteusNfCVrjAkffLK1sbh4hTLM8tl annotation.zip
Processing file 1Bod3MrbdCRiSvpR7NSOYi-rF3-ushwtj data.zip
Processing file 1zm

## 4. Verify the exact 2027 Track-2 data contract

In [7]:
!python {PIPE}/scripts/01_verify_official_data.py --project-root {PROJECT}


python3: can't open file '/content/upar_pipeline/scripts/01_verify_official_data.py': [Errno 2] No such file or directory


Just accessing the folder delete if need be

In [9]:

import os

for base in [
    "/content/UPAR-Challenge-2027",
    "/content/upar_pipeline",
    "/content/drive/MyDrive/UPAR2027_Track2_CodaBenchAligned",
]:
    print(f"\n{'='*65}\n{base}")
    if os.path.exists(base):
        for root, dirs, files in os.walk(base):
            depth = root[len(base):].count(os.sep)
            if depth >= 2:
                dirs[:] = []
            print("  " * depth + os.path.basename(root) + "/")
            for f in files:
                if (
                    "verify" in f.lower()
                    or "best_dev" in f.lower()
                    or "training_config" in f.lower()
                    or f.endswith(".zip")
                ):
                    print("  " * (depth + 1) + f)
    else:
        print("PATH DOES NOT EXIST")



/content/UPAR-Challenge-2027
UPAR-Challenge-2027/
  examples/
    task1/
    task2/
  .git/
    refs/
    branches/
    objects/
    hooks/
    info/
    logs/
  data/
    market_1501.zip
    annotations/
    PA100k/
      data.zip
      annotation.zip
    Market1501/
    PETA/
      peta.zip

/content/upar_pipeline
upar_pipeline/
  upar_track2_codabench_aligned_v2.zip

/content/drive/MyDrive/UPAR2027_Track2_CodaBenchAligned
PATH DOES NOT EXIST


In [15]:

import os
import torch

checkpoint_path = (
    "/content/drive/MyDrive/convnext_tiny_hybrid/best_dev.pt"
)

print("Checkpoint size (MB):",
      round(os.path.getsize(checkpoint_path) / (1024**2), 2))

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", list(checkpoint.keys()))

    for key in ("config", "args", "epoch", "best_metric", "metrics"):
        if key in checkpoint:
            print(f"\n{key}:", checkpoint[key])


Checkpoint size (MB): 108.61

Checkpoint type: <class 'dict'>
Checkpoint keys: ['model_state', 'config', 'epoch', 'attribute_names', 'attribute_weights']

config: {'project_root': '/content/UPAR-Challenge-2027', 'workers': 4, 'dropout': 0.2, 'pos_weight_clip': 20.0, 'label_smoothing': 0.05, 'lr_patience': 4, 'temperature': 0.07, 'madmin_subset_queries': 256, 'seed': 2027, 'name': 'convnext_tiny_hybrid', 'backbone': 'convnext_tiny', 'embed_dim': 256, 'domain_balanced': True, 'epochs': 8, 'batch_size': 64, 'lr': 0.0001, 'weight_decay': 0.0005, 'lambda_contrastive': 0.3, 'output': '/content/drive/MyDrive/UPAR2027_Track2_CodaBenchAligned/convnext_tiny_hybrid'}

epoch: 8


In [17]:

import os
import zipfile

zip_path = "/content/upar_pipeline/upar_track2_codabench_aligned_v2.zip"
extract_dir = "/content/upar_pipeline/extracted"

print("ZIP exists:", os.path.exists(zip_path))

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:
        print("\nZIP contains these files:")
        for name in z.namelist():
            if (
                name.endswith("models.py")
                or name.endswith("retrieval.py")
                or name.endswith("train_models.py")
                or name.endswith("run.py")
                or "graph" in name.lower()
                or "rerank" in name.lower()
            ):
                print(name)

    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)

    print("\nExtracted successfully to:", extract_dir)
else:
    print("ZIP not found. Check /content/upar_pipeline/")


ZIP exists: True

ZIP contains these files:
submission_template/run.py
scripts/03_train_models.py
src/upar/retrieval.py
src/upar/models.py

Extracted successfully to: /content/upar_pipeline/extracted


In [18]:

from pathlib import Path

root = Path("/content/upar_pipeline/extracted")

files_to_check = [
    root / "src/upar/models.py",
    root / "src/upar/retrieval.py",
]

for path in files_to_check:
    print("\n" + "=" * 70)
    print(f"FILE: {path}")
    print("=" * 70)

    if path.exists():
        print(path.read_text())
    else:
        print("NOT FOUND")



FILE: /content/upar_pipeline/extracted/src/upar/models.py

from __future__ import annotations
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import (
    resnet50, ResNet50_Weights,
    convnext_tiny, ConvNeXt_Tiny_Weights
)

class UPARNet(nn.Module):
    def __init__(self, backbone, embed_dim=0, pretrained=True, dropout=.2):
        super().__init__()
        self.backbone_name = backbone
        self.embed_dim = int(embed_dim)
        if backbone == "resnet50":
            m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
            dim = m.fc.in_features
            m.fc = nn.Identity()
        elif backbone == "convnext_tiny":
            m = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None)
            dim = m.classifier[2].in_features
            m.classifier[2] = nn.Identity()
        else:
            raise ValueError(backbone)
        self.backbone = m
        self.attr_he

## 5. Save runs to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RUNS = "/content/drive/MyDrive/UPAR2027_Track2_CodaBenchAligned"
os.makedirs(RUNS, exist_ok=True)
print("Results:", RUNS)


Mounted at /content/drive
Results: /content/drive/MyDrive/UPAR2027_Track2_CodaBenchAligned


## 6. Evaluate the organizer-style prior baseline

In [ ]:
!python {PIPE}/scripts/02_official_prior_baseline.py \
  --project-root {PROJECT} \
  --output {RUNS}/official_prior_baseline.json


{
  "mADM": 0.1133234861459536,
  "mAP": 0.0006305639428390385,
  "R1": 0.00028885037550548814,
  "n_queries": 3462
}


## 7. Train all development models

- ResNet50 baseline
- ConvNeXt-Tiny baseline
- ConvNeXt-Tiny hybrid retrieval model

All learnable models use allowed ImageNet pretraining and organizer-provided UPAR data only.


In [ ]:
!python {PIPE}/scripts/03_train_models.py \
  --project-root {PROJECT} \
  --runs {RUNS}



### TRAINING resnet50_baseline
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100% 97.8M/97.8M [00:00<00:00, 198MB/s]
/content/upar_pipeline/src/upar/train_eval.py:71: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=GradScaler(enabled=device.type=="cuda")
resnet50_baseline epoch 1/6:   0% 0/1526 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-pa

In [ ]:
import os

os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

print("Checkpoint compatibility fix enabled.")

Checkpoint compatibility fix enabled.


## 8. Full validation and model selection using mADM

In [ ]:
from pathlib import Path

p = Path(PIPE) / "src/upar/train_eval.py"
text = p.read_text()

# Fix validation DataLoader hang
text = text.replace(
    'dl=DataLoader(ds,batch_size=128,shuffle=False,num_workers=4,pin_memory=True)',
    'dl=DataLoader(ds,batch_size=128,shuffle=False,num_workers=0,pin_memory=True)'
)

# Make checkpoint loading explicit for newer PyTorch
text = text.replace(
    'torch.load(checkpoint,map_location="cpu")',
    'torch.load(checkpoint,map_location="cpu",weights_only=False)'
)

p.write_text(text)

print("Patched successfully.")

# Confirm the important lines
for i, line in enumerate(p.read_text().splitlines(), 1):
    if 135 <= i <= 146:
        print(f"{i}: {line}")

Patched successfully.
135: 
136: def full_validate(checkpoint, project_root, output_json):
137:     root=Path(project_root); ckpt=torch.load(checkpoint,map_location="cpu",weights_only=False)
138:     model=from_config(ckpt["config"],pretrained=False)
139:     model.load_state_dict(ckpt["model_state"]); device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
140:     model.to(device).eval()
141: 
142:     base=root/"data/annotations/task2/val"
143:     ds=UPARDataset(root,base,eval_transform())
144:     dl=DataLoader(ds,batch_size=128,shuffle=False,num_workers=0,pin_memory=True)
145:     gt,p,z=infer(model,dl,device)
146:     q,_=read_queries_csv(base/"queries.csv")


In [ ]:
!python {PIPE}/scripts/04_validate_select.py \
  --project-root {PROJECT} \
  --runs {RUNS}

## 9. Final train on organizer train + validation

In [ ]:
!python {PIPE}/scripts/05_final_train.py \
  --project-root {PROJECT} \
  --runs {RUNS}


## 10. Build and smoke-test the CodaBench submission

In [ ]:
!python {PIPE}/scripts/06_build_submission.py \
  --runs {RUNS} \
  --output {RUNS}/codabench_submission

!python {PIPE}/scripts/07_smoke_test.py \
  --submission-dir {RUNS}/codabench_submission


## 11. Download the final submission

In [ ]:
from google.colab import files
files.download(RUNS + "/codabench_submission.zip")


Additional - Load the trained model  This is where I did most of the work ignore these cells since they include Graph reranking which tanked for Convnext. The cells after Catboost are relevant thats where the madm is at 67%

In [19]:

import sys
from pathlib import Path
import torch

PIPELINE = Path("/content/upar_pipeline/extracted")
SRC = PIPELINE / "src"
CHECKPOINT_PATH = (
    "/content/drive/MyDrive/convnext_tiny_hybrid/best_dev.pt"
)

assert SRC.exists(), f"Source folder not found: {SRC}"
assert Path(CHECKPOINT_PATH).stat().st_size > 0, "Checkpoint is empty"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from upar.models import from_config

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

config = checkpoint["config"]
assert config["name"] == "convnext_tiny_hybrid", config["name"]
assert config["backbone"] == "convnext_tiny", config["backbone"]
assert config["embed_dim"] == 256, config["embed_dim"]

# Construct the exact architecture without downloading pretrained weights.
model = from_config(config, pretrained=False)

# Load the teammate's already-trained parameters.
result = model.load_state_dict(checkpoint["model_state"], strict=True)
print("Checkpoint loaded:", result)

model = model.to(device).eval()

# Smoke-test the forward pass using a dummy input.
with torch.inference_mode():
    dummy = torch.zeros(1, 3, 224, 224, device=device)
    logits, image_z = model(dummy)
    query_z = model.encode_query(torch.zeros(1, 40, device=device))

print("Attribute logits:", tuple(logits.shape))
print("Image embedding:", tuple(image_z.shape))
print("Query embedding:", tuple(query_z.shape))
print("Model ready for inference.")


Device: cuda
Checkpoint loaded: <All keys matched successfully>
Attribute logits: (1, 40)
Image embedding: (1, 256)
Query embedding: (1, 256)
Model ready for inference.


Find the evaluation pipeline and graph code

In [20]:

from pathlib import Path

roots = [
    Path("/content/upar_pipeline/extracted"),
    Path("/content/UPAR-Challenge-2027"),
]

keywords = (
    "dataset", "metric", "eval", "retrieval", "graph",
    "rerank", "run.py", "predict", "inference", "cache"
)

for base in roots:
    print(f"\n{'='*70}\n{base}")
    if not base.exists():
        print("Not found")
        continue

    for p in sorted(base.rglob("*")):
        if not p.is_file():
            continue
        if any(part in {".git", "__pycache__"} for part in p.parts):
            continue
        if p.suffix in {".py", ".ipynb", ".json", ".yaml", ".yml"}:
            if any(k in p.name.lower() for k in keywords):
                print(p)



/content/upar_pipeline/extracted
/content/upar_pipeline/extracted/src/upar/metrics.py
/content/upar_pipeline/extracted/src/upar/retrieval.py
/content/upar_pipeline/extracted/src/upar/train_eval.py
/content/upar_pipeline/extracted/submission_template/run.py

/content/UPAR-Challenge-2027
/content/UPAR-Challenge-2027/download_datasets.py
/content/UPAR-Challenge-2027/examples/task1/sample_code_submission/run.py
/content/UPAR-Challenge-2027/examples/task2/sample_code_submission/run.py


INspect evaluation and submission interface

In [21]:

from pathlib import Path

root = Path("/content/upar_pipeline/extracted")

files_to_read = [
    root / "src/upar/train_eval.py",
    root / "src/upar/metrics.py",
    root / "submission_template/run.py",
]

for path in files_to_read:
    print("\n" + "=" * 75)
    print("FILE:", path)
    print("=" * 75)

    if path.exists():
        print(path.read_text())
    else:
        print("FILE NOT FOUND")



FILE: /content/upar_pipeline/extracted/src/upar/train_eval.py

from __future__ import annotations
from pathlib import Path
import json, random, copy

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

from .data import (
    UPARDataset, train_transform, eval_transform,
    compute_pos_weight, inverse_sqrt_attr_weights, domain_balanced_sampler,
    read_queries_csv, read_ids_csv
)
from .losses import smooth_binary_targets, multipositive_infonce
from .metrics import classification_metrics
from .models import UPARNet, from_config
from .retrieval import eval_retrieval, sweep_configs


def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def infer(model, loader, device):
    model.eval()
    probs=[]; labels=[]; zs=[]
    for b in tqdm(loader,desc

Establish ConvNeXt hyrbid baseline

In [22]:

from pathlib import Path

root = Path("/content/upar_pipeline/extracted")

# Inspect how images, labels, and query IDs are loaded.
data_file = root / "src/upar/data.py"
print("=" * 70)
print("DATA LOADER")
print("=" * 70)
print(data_file.read_text() if data_file.exists() else "data.py not found")

# Check the official repository's Task 2 validation annotations.
repo = Path("/content/UPAR-Challenge-2027")
for path in [
    repo / "data/annotations/task2/val/queries.csv",
    repo / "data/annotations/task2/val/ids.csv",
    repo / "data/annotations/task2/train",
]:
    print("\n", path, "| exists:", path.exists())
    if path.is_file():
        print("Size:", path.stat().st_size, "bytes")


DATA LOADER

from __future__ import annotations
from pathlib import Path
from typing import Sequence

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, WeightedRandomSampler
from torchvision import transforms

from .constants import NUM_ATTRIBUTES, INPUT_HEIGHT, INPUT_WIDTH, IMAGENET_MEAN, IMAGENET_STD


def _clean(s):
    return str(s).strip().lstrip("#").strip()


def read_gt_csv(path):
    """2027 organizer format: # image,<40 binary attrs>."""
    df = pd.read_csv(path)
    df.columns = [_clean(c) for c in df.columns]
    if df.shape[1] != 1 + NUM_ATTRIBUTES:
        raise ValueError(f"{path}: expected 41 columns, got {df.shape[1]}")
    paths = df.iloc[:, 0].astype(str).tolist()
    labels = df.iloc[:, 1:].apply(pd.to_numeric, errors="raise").to_numpy(np.float32)
    names = [_clean(c) for c in df.columns[1:]]
    return paths, labels, names


def read_ids_csv(path):
    """2027 organizer format: image_path,zero_based_q

Validation Inference

In [23]:

import sys
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader

PIPELINE = Path("/content/upar_pipeline/extracted")
REPO = Path("/content/UPAR-Challenge-2027")
SRC = str(PIPELINE / "src")

if SRC not in sys.path:
    sys.path.insert(0, SRC)

from upar.data import (
    UPARDataset, eval_transform, read_queries_csv, read_ids_csv
)
from upar.train_eval import infer, encode_queries
from upar.retrieval import eval_retrieval

# Reuse the already-loaded model and checkpoint.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

val_base = REPO / "data/annotations/task2/val"
dataset = UPARDataset(REPO, val_base, eval_transform())
loader = DataLoader(
    dataset,
    batch_size=96,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Validation gallery images:", len(dataset))
print("Device:", device)

# Frozen model inference only.
gt_labels, probs, image_z = infer(model, loader, device)

queries, query_names = read_queries_csv(val_base / "queries.csv")
_, image_query_ids = read_ids_csv(val_base / "ids.csv")
query_z = encode_queries(model, queries, device)

assert len(gt_labels) == len(probs) == len(image_query_ids)
assert probs.shape[1] == 40
assert image_z is not None and image_z.shape[1] == 256
assert query_z is not None and query_z.shape == (len(queries), 256)

# Match the deterministic subset used by the training evaluator.
rng = np.random.default_rng(2027)
subset = np.sort(
    rng.choice(len(queries), size=min(256, len(queries)), replace=False)
)

baseline = eval_retrieval(
    probs, queries, gt_labels, image_query_ids,
    query_indices=subset,
    mode="l1",
    weights=np.ones(40, dtype=np.float32),
    image_z=image_z,
    query_z=query_z,
    alpha=1.0,  # Official attribute-only baseline
)

print("\nBASELINE — hybrid ConvNeXt, graph OFF")
print(baseline)

# Cache all arrays for repeated graph/reranking experiments.
cache_dir = Path("/content/drive/MyDrive/convnext_tiny_hybrid")
cache_dir.mkdir(parents=True, exist_ok=True)
cache_path = cache_dir / "validation_cache.npz"

np.savez_compressed(
    cache_path,
    gt_labels=gt_labels,
    probs=probs,
    image_z=image_z,
    queries=queries,
    query_z=query_z,
    image_query_ids=image_query_ids,
    subset=subset,
    attribute_names=np.asarray(query_names),
)

print("\nCached validation predictions:", cache_path)
print("No training was performed.")


Validation gallery images: 33407
Device: cuda



BASELINE — hybrid ConvNeXt, graph OFF
{'mADM': 0.6231123875339057, 'mAP': 0.18460191572855036, 'R1': 0.23828125, 'n_queries': 256}

Cached validation predictions: /content/drive/MyDrive/convnext_tiny_hybrid/validation_cache.npz
No training was performed.


Verify Evaluator and Cache before graph construction

In [26]:

import inspect
import numpy as np
from pathlib import Path

cache_path = Path(
    "/content/drive/MyDrive/convnext_tiny_hybrid/validation_cache.npz"
)

assert cache_path.exists(), f"Cache not found: {cache_path}"

with np.load(cache_path, allow_pickle=False) as cache:
    for key in cache.files:
        arr = cache[key]
        print(f"{key:20s} shape={str(arr.shape):18s} dtype={arr.dtype}")

    probs = cache["probs"]
    image_z = cache["image_z"]
    queries = cache["queries"]
    query_z = cache["query_z"]
    gt_labels = cache["gt_labels"]
    image_query_ids = cache["image_query_ids"]
    subset = cache["subset"]

print("\nFinite predictions:", np.isfinite(probs).all())
print("Finite embeddings:", np.isfinite(image_z).all())
print("Query subset size:", len(subset))
print("Embedding dimension:", image_z.shape[1])

print("\n===== REPOSITORY EVALUATOR =====")
print(inspect.getsource(eval_retrieval))


gt_labels            shape=(33407, 40)        dtype=float32
probs                shape=(33407, 40)        dtype=float32
image_z              shape=(33407, 256)       dtype=float32
queries              shape=(3462, 40)         dtype=float32
query_z              shape=(3462, 256)        dtype=float32
image_query_ids      shape=(33407,)           dtype=int64
subset               shape=(256,)             dtype=int64
attribute_names      shape=(40,)              dtype=<U30

Finite predictions: True
Finite embeddings: True
Query subset size: 256
Embedding dimension: 256

===== REPOSITORY EVALUATOR =====
def eval_retrieval(
    probs, queries, gt_labels, image_query_ids,
    query_indices=None, mode="l1", weights=None,
    image_z=None, query_z=None, alpha=1.0,
):
    if query_indices is None:
        query_indices=np.arange(len(queries),dtype=np.int64)
    adm_vals=[]; ap_vals=[]; r1=[]
    chunk=32
    for s in range(0,len(query_indices),chunk):
        idx=query_indices[s:s+chunk]
        

** ========== mADM improved ========== **

# **DIFFERENT APPROACH CAT BOOST + Ensemble**

In [32]:

# CELL 1 — Check the cached ensemble inputs

import numpy as np

required = [
    "probs", "queries", "image_z", "query_z",
    "gt_labels", "image_query_ids", "subset",
    "l1_distances", "evaluate_score_matrix"
]

missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f"Missing variables/functions: {missing}")

print("All required variables found!\n")

print("Gallery attribute probabilities:", probs.shape)
print("Query attribute labels:", queries.shape)
print("Gallery embeddings:", image_z.shape)
print("Query embeddings:", query_z.shape)
print("Evaluation query subset:", subset.shape)

assert probs.shape[0] == image_z.shape[0]
assert query_z.shape[0] == queries.shape[0]
assert image_z.shape[1] == query_z.shape[1]
assert len(subset) > 0
assert np.isfinite(probs).all()
assert np.isfinite(image_z).all()
assert np.isfinite(query_z).all()

print("\nFinite values: PASS")
print("Embedding dimensions match: PASS")
print("Ready to test the ensemble.")


All required variables found!

Gallery attribute probabilities: (33407, 40)
Query attribute labels: (3462, 40)
Gallery embeddings: (33407, 256)
Query embeddings: (3462, 256)
Evaluation query subset: (256,)

Finite values: PASS
Embedding dimensions match: PASS
Ready to test the ensemble.


Ensemble sweep

In [33]:

# CELL 2 — Attribute + embedding ensemble sweep
# Uses the repository's own evaluator and the same 256-query subset.

import numpy as np
import pandas as pd
from upar.retrieval import eval_retrieval

weights = np.ones(40, dtype=np.float32)

# alpha = 1.0 means attributes only.
# alpha = 0.0 means embeddings only.
# Intermediate values blend both signals.
alphas = [1.0, 0.95, 0.90, 0.85, 0.75, 0.60, 0.50, 0.35, 0.25, 0.0]

results = []

for alpha in alphas:
    result = eval_retrieval(
        probs=probs,
        queries=queries,
        gt_labels=gt_labels,
        image_query_ids=image_query_ids,
        query_indices=subset,
        mode="l1",
        weights=weights,
        image_z=image_z,
        query_z=query_z,
        alpha=alpha,
    )

    results.append({
        "alpha": alpha,
        "attribute_weight_pct": round(alpha * 100, 1),
        "embedding_weight_pct": round((1 - alpha) * 100, 1),
        "mADM": result["mADM"],
        "mAP": result["mAP"],
        "R1": result["R1"],
    })

results_df = pd.DataFrame(results).sort_values(
    "mADM", ascending=False
).reset_index(drop=True)

baseline_madm = results_df.loc[
    results_df["alpha"] == 1.0, "mADM"
].iloc[0]

results_df["delta_mADM_pp"] = (
    results_df["mADM"] - baseline_madm
) * 100

print("ENSEMBLE RESULTS — sorted by mADM")
display(results_df.round(5))

best = results_df.iloc[0]
print("\nBEST RESULT")
print(f"Attribute weight: {best['attribute_weight_pct']}%")
print(f"Embedding weight: {best['embedding_weight_pct']}%")
print(f"mADM: {best['mADM']:.5f}")
print(f"Change vs baseline: {best['delta_mADM_pp']:+.3f} percentage points")


ENSEMBLE RESULTS — sorted by mADM


,alpha,attribute_weight_pct,embedding_weight_pct,mADM,mAP,R1,delta_mADM_pp
0,0.60,60.0,40.0,0.67207,0.24643,0.28516,4.89608
1,0.50,50.0,50.0,0.67087,0.25189,0.28516,4.77562
2,0.35,35.0,65.0,0.66849,0.25879,0.28516,4.53774
3,0.25,25.0,75.0,0.66813,0.26261,0.28516,4.50164
4,0.75,75.0,25.0,0.66736,0.23378,0.28125,4.42516
5,0.00,0.0,100.0,0.65954,0.26336,0.30078,3.64268
6,0.85,85.0,15.0,0.65937,0.21889,0.27344,3.62545
7,0.90,90.0,10.0,0.65145,0.21038,0.26172,2.83378
8,0.95,95.0,5.0,0.64032,0.19878,0.25000,1.72100
9,1.00,100.0,0.0,0.62311,0.18460,0.23828,0.00000



BEST RESULT
Attribute weight: 60.0%
Embedding weight: 40.0%
mADM: 0.67207
Change vs baseline: +4.896 percentage points


Fine grained alpha sweep

In [34]:

# CELL 3 — Fine sweep around the current best alpha = 0.60

import pandas as pd
import numpy as np
from upar.retrieval import eval_retrieval

fine_alphas = [0.52, 0.54, 0.56, 0.58, 0.60,
               0.62, 0.64, 0.66, 0.68]

fine_results = []

for alpha in fine_alphas:
    result = eval_retrieval(
        probs=probs,
        queries=queries,
        gt_labels=gt_labels,
        image_query_ids=image_query_ids,
        query_indices=subset,
        mode="l1",
        weights=np.ones(40, dtype=np.float32),
        image_z=image_z,
        query_z=query_z,
        alpha=alpha,
    )

    fine_results.append({
        "alpha": alpha,
        "attribute_pct": round(100 * alpha),
        "embedding_pct": round(100 * (1 - alpha)),
        "mADM": result["mADM"],
        "mAP": result["mAP"],
        "R1": result["R1"],
    })

fine_df = pd.DataFrame(fine_results).sort_values(
    "mADM", ascending=False
).reset_index(drop=True)

print("FINE SWEEP — best first")
display(fine_df.round(5))

best_alpha = float(fine_df.iloc[0]["alpha"])
best_madm = float(fine_df.iloc[0]["mADM"])

print(f"\nBest alpha: {best_alpha:.2f}")
print(f"Attribute weight: {best_alpha:.0%}")
print(f"Embedding weight: {1-best_alpha:.0%}")
print(f"Best mADM: {best_madm:.5f}")


FINE SWEEP — best first


,alpha,attribute_pct,embedding_pct,mADM,mAP,R1
0,0.64,64,36,0.67273,0.24472,0.28125
1,0.62,62,38,0.67210,0.24549,0.28125
2,0.60,60,40,0.67207,0.24643,0.28516
3,0.56,56,44,0.67184,0.24862,0.28516
4,0.58,58,42,0.67172,0.24727,0.28516
5,0.66,66,34,0.67168,0.24202,0.27344
6,0.54,54,46,0.67159,0.24963,0.28516
7,0.52,52,48,0.67082,0.25069,0.28516
8,0.68,68,32,0.67049,0.24040,0.27344



Best alpha: 0.64
Attribute weight: 64%
Embedding weight: 36%
Best mADM: 0.67273


Evaluate all the queries

In [35]:

# CELL 4 — Full validation evaluation of the ensemble
# Frozen model; no training or image inference required.

import numpy as np
import pandas as pd
from upar.retrieval import eval_retrieval

full_indices = np.arange(len(queries), dtype=np.int64)
attr_weights = np.ones(40, dtype=np.float32)

# Compare the original baseline with nearby ensemble weights.
alphas_to_validate = [1.00, 0.64, 0.60, 0.68]

full_results = []

for alpha in alphas_to_validate:
    result = eval_retrieval(
        probs=probs,
        queries=queries,
        gt_labels=gt_labels,
        image_query_ids=image_query_ids,
        query_indices=full_indices,
        mode="l1",
        weights=attr_weights,
        image_z=image_z,
        query_z=query_z,
        alpha=alpha,
    )

    full_results.append({
        "alpha": alpha,
        "attribute_pct": round(100 * alpha),
        "embedding_pct": round(100 * (1 - alpha)),
        "mADM": result["mADM"],
        "mAP": result["mAP"],
        "R1": result["R1"],
        "n_queries": result["n_queries"],
    })

full_df = pd.DataFrame(full_results).sort_values(
    "mADM", ascending=False
).reset_index(drop=True)

baseline_row = full_df.loc[full_df["alpha"] == 1.00].iloc[0]
full_df["delta_mADM_pp"] = (
    full_df["mADM"] - baseline_row["mADM"]
) * 100

print("FULL VALIDATION RESULTS — best first")
display(full_df.round(5))

best_full = full_df.iloc[0]
print("\nBest full-validation configuration:")
print(f"Attribute weight: {best_full['attribute_pct']}%")
print(f"Embedding weight: {best_full['embedding_pct']}%")
print(f"mADM: {best_full['mADM']:.5f}")
print(f"Change vs baseline: {best_full['delta_mADM_pp']:+.3f} percentage points")
print(f"Queries evaluated: {int(best_full['n_queries'])}")


FULL VALIDATION RESULTS — best first


,alpha,attribute_pct,embedding_pct,mADM,mAP,R1,n_queries,delta_mADM_pp
0,0.60,60,40,0.67258,0.24815,0.27701,3462,6.08244
1,0.64,64,36,0.67244,0.24564,0.27556,3462,6.06808
2,0.68,68,32,0.67146,0.24135,0.27239,3462,5.96998
3,1.00,100,0,0.61176,0.16427,0.21144,3462,0.00000



Best full-validation configuration:
Attribute weight: 60.0%
Embedding weight: 40.0%
mADM: 0.67258
Change vs baseline: +6.082 percentage points
Queries evaluated: 3462


L1 VS NLL

In [36]:

import numpy as np
import pandas as pd
from upar.retrieval import eval_retrieval

full_indices = np.arange(len(queries), dtype=np.int64)
attr_weights = np.ones(40, dtype=np.float32)

experiments = [
    ("l1", 1.00),
    ("l1", 0.60),
    ("nll", 1.00),
    ("nll", 0.60),
]

rows = []

for mode, alpha in experiments:
    result = eval_retrieval(
        probs=probs,
        queries=queries,
        gt_labels=gt_labels,
        image_query_ids=image_query_ids,
        query_indices=full_indices,
        mode=mode,
        weights=attr_weights,
        image_z=image_z,
        query_z=query_z,
        alpha=alpha,
    )

    rows.append({
        "mode": mode,
        "alpha": alpha,
        "attribute_pct": round(alpha * 100),
        "embedding_pct": round((1 - alpha) * 100),
        "mADM": result["mADM"],
        "mAP": result["mAP"],
        "R1": result["R1"],
        "n_queries": result["n_queries"],
    })

results_df = pd.DataFrame(rows).sort_values(
    "mADM", ascending=False
).reset_index(drop=True)

print("L1 vs NLL — FULL VALIDATION")
display(results_df.round(5))


L1 vs NLL — FULL VALIDATION


,mode,alpha,attribute_pct,embedding_pct,mADM,mAP,R1,n_queries
0,l1,0.6,60,40,0.67258,0.24815,0.27701,3462
1,nll,0.6,60,40,0.66244,0.23864,0.26950,3462
2,nll,1.0,100,0,0.63011,0.20155,0.24003,3462
3,l1,1.0,100,0,0.61176,0.16427,0.21144,3462


Inspect attribute usefullness

In [38]:

import numpy as np
import pandas as pd
from pathlib import Path

# Recover attribute names if they are missing
if "attribute_names" not in globals():
    candidates = [
        Path("/content/upar_pipeline/extracted"),
        Path("/content/UPAR-Challenge-2027"),
    ]

    found_names = None

    for root in candidates:
        if not root.exists():
            continue

        for csv_path in root.rglob("*.csv"):
            try:
                df = pd.read_csv(csv_path, nrows=2)
                cols = list(df.columns)

                # Look for a metadata file with the 40 label columns.
                if len(cols) >= 40 and any(
                    "attribute" in c.lower() or "label" in c.lower()
                    for c in cols
                ):
                    print("Possible metadata file:", csv_path)
            except Exception:
                pass

    # Safe fallback: keep the analysis running without guessing names.
    attribute_names = np.array(
        [f"attribute_{i}" for i in range(gt_labels.shape[1])]
    )
    print("Using numbered attribute names for now.")

assert gt_labels.ndim == 2
assert gt_labels.shape[1] == len(attribute_names)

prevalence = gt_labels.mean(axis=0)

attribute_stats = pd.DataFrame({
    "attribute": attribute_names,
    "prevalence": prevalence,
    "positive_images": gt_labels.sum(axis=0).astype(int),
    "negative_images": (len(gt_labels) - gt_labels.sum(axis=0)).astype(int),
})

attribute_stats["rarity"] = np.minimum(
    attribute_stats["prevalence"],
    1 - attribute_stats["prevalence"]
)

display(
    attribute_stats.sort_values("rarity")
    .reset_index(drop=True)
    .round(4)
)


Using numbered attribute names for now.


,attribute,prevalence,positive_images,negative_images,rarity
0,attribute_28,0.0008,26,33381,0.0008
1,attribute_31,0.0017,56,33351,0.0017
2,attribute_26,0.0021,70,33337,0.0021
3,attribute_29,0.0039,130,33277,0.0039
4,attribute_13,0.0058,194,33213,0.0058
5,attribute_14,0.0105,351,33056,0.0105
6,attribute_0,0.0110,368,33039,0.0110
7,attribute_6,0.0126,421,32986,0.0126
8,attribute_24,0.0150,500,32907,0.0150
9,attribute_27,0.0155,517,32890,0.0155


Test prediction based attribute weighting

In [39]:

import numpy as np
import pandas as pd
from upar.retrieval import eval_retrieval

# Keep our best blend fixed.
ALPHA = 0.60
full_indices = np.arange(len(queries), dtype=np.int64)
n_attributes = probs.shape[1]

# Prediction variability, calculated without ground-truth labels.
pred_std = np.std(probs, axis=0).astype(np.float64)
eps = 1e-6

# Candidate weighting schemes.
weight_schemes = {
    "uniform": np.ones(n_attributes),
    "variation_mild": (pred_std + eps) ** 0.5,
    "variation_strong": pred_std + eps,
    "reverse_variation_mild": 1.0 / np.sqrt(pred_std + eps),
}

rows = []

for scheme_name, raw_weights in weight_schemes.items():
    # Normalize so average attribute weight is 1.
    weights = raw_weights / np.mean(raw_weights)
    weights = weights.astype(np.float32)

    result = eval_retrieval(
        probs=probs,
        queries=queries,
        gt_labels=gt_labels,
        image_query_ids=image_query_ids,
        query_indices=full_indices,
        mode="l1",
        weights=weights,
        image_z=image_z,
        query_z=query_z,
        alpha=ALPHA,
    )

    rows.append({
        "scheme": scheme_name,
        "min_weight": float(weights.min()),
        "max_weight": float(weights.max()),
        "mADM": result["mADM"],
        "mAP": result["mAP"],
        "R1": result["R1"],
        "n_queries": result["n_queries"],
    })

weight_results = pd.DataFrame(rows).sort_values(
    "mADM", ascending=False
).reset_index(drop=True)

baseline_madm = weight_results.loc[
    weight_results["scheme"] == "uniform", "mADM"
].iloc[0]

weight_results["delta_vs_uniform_pp"] = (
    weight_results["mADM"] - baseline_madm
) * 100

display(weight_results.round(5))


,scheme,min_weight,max_weight,mADM,mAP,R1,n_queries,delta_vs_uniform_pp
0,variation_mild,0.43085,1.49600,0.67332,0.24540,0.27296,3462,0.07329
1,uniform,1.00000,1.00000,0.67258,0.24815,0.27701,3462,0.00000
2,variation_strong,0.16777,2.02261,0.67060,0.24047,0.26950,3462,-0.19824
3,reverse_variation_mild,0.59225,2.05640,0.66539,0.24737,0.27816,3462,-0.71935


Final Hybrid submission

In [40]:

from pathlib import Path
import os
import sys
import json
import shutil
import zipfile
import subprocess
import numpy as np
import torch
import torch.nn as nn

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------
PIPE = Path("/content/upar_pipeline/extracted")
SRC = PIPE / "src"
TEMPLATE = PIPE / "submission_template"
CHECKPOINT = Path(
    "/content/drive/MyDrive/convnext_tiny_hybrid/best_dev.pt"
)
OUT = Path(
    "/content/drive/MyDrive/convnext_tiny_hybrid/final_submission"
)
ZIP_PATH = OUT.with_suffix(".zip")

assert CHECKPOINT.exists(), f"Checkpoint not found: {CHECKPOINT}"
assert (SRC / "upar/models.py").exists(), "Model source not found."
assert (TEMPLATE / "run.py").exists(), "Submission runner not found."
assert (TEMPLATE / "metadata.yaml").exists(), "Submission metadata not found."

sys.path.insert(0, str(SRC))
from upar.models import from_config
from upar.constants import (
    INPUT_HEIGHT, INPUT_WIDTH, IMAGENET_MEAN, IMAGENET_STD
)

# --------------------------------------------------
# 2. Load the checkpoint without training
# --------------------------------------------------
try:
    ckpt = torch.load(
        CHECKPOINT, map_location="cpu", weights_only=True
    )
except Exception:
    # Compatibility fallback for checkpoints containing
    # Python metadata as well as tensors.
    ckpt = torch.load(
        CHECKPOINT, map_location="cpu", weights_only=False
    )

print("Checkpoint keys:", list(ckpt.keys()))

config = ckpt["config"]
state = ckpt.get("model_state", ckpt.get("model_state_dict"))
assert state is not None, "Could not find model weights in checkpoint."

model = from_config(config, pretrained=False)
model.load_state_dict(state, strict=True)
model.eval()

attribute_names = ckpt.get("attribute_names")
assert attribute_names is not None and len(attribute_names) == 40
assert model.query_head is not None, (
    "This checkpoint has no query encoder; hybrid retrieval cannot run."
)

print("Backbone:", config["backbone"])
print("Embedding dimension:", config.get("embed_dim"))
print("Attributes:", len(attribute_names))
print("Checkpoint loaded successfully.")

# --------------------------------------------------
# 3. Export the image and query encoders
# --------------------------------------------------
class ImageExport(nn.Module):
    def __init__(self, wrapped_model):
        super().__init__()
        self.model = wrapped_model

    def forward(self, images):
        logits, embeddings = self.model(images)
        if embeddings is None:
            embeddings = torch.empty(
                (images.shape[0], 0),
                dtype=logits.dtype,
                device=logits.device,
            )
        return logits, embeddings


class QueryExport(nn.Module):
    def __init__(self, wrapped_model):
        super().__init__()
        self.model = wrapped_model

    def forward(self, query_attributes):
        return self.model.encode_query(query_attributes)


shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True)
(OUT / "assets").mkdir()
(OUT / "vendor").mkdir()

image_export = torch.jit.trace(
    ImageExport(model).eval(),
    torch.randn(1, 3, INPUT_HEIGHT, INPUT_WIDTH),
)
image_export.save(str(OUT / "assets/image_model.ts"))

query_export = torch.jit.trace(
    QueryExport(model).eval(),
    torch.zeros(1, 40),
)
query_export.save(str(OUT / "assets/query_model.ts"))

# --------------------------------------------------
# 4. Write the chosen retrieval configuration
# --------------------------------------------------
cfg = {
    "attribute_names": list(attribute_names),
    "attribute_weights": [1.0] * 40,
    "mode": "l1",
    "weights": "uniform",
    "alpha": 0.60,
    "height": INPUT_HEIGHT,
    "width": INPUT_WIDTH,
    "mean": IMAGENET_MEAN,
    "std": IMAGENET_STD,
    "batch_size": 64,
    "torch_export_version": torch.__version__,
}
(OUT / "assets/config.json").write_text(
    json.dumps(cfg, indent=2)
)

shutil.copy2(TEMPLATE / "run.py", OUT / "run.py")
shutil.copy2(TEMPLATE / "metadata.yaml", OUT / "metadata.yaml")

# Include Pillow for an offline submission environment.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--no-deps",
    "--target", str(OUT / "vendor"), "Pillow==10.4.0"
])

# --------------------------------------------------
# 5. Package it
# --------------------------------------------------
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH, "w", zipfile.ZIP_DEFLATED
) as archive:
    for path in OUT.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(OUT))

print("\nSUBMISSION PACKAGE BUILT")
print("Folder:", OUT)
print("ZIP:", ZIP_PATH)
print("ZIP size (MB):", round(ZIP_PATH.stat().st_size / 1024**2, 2))
print("Selected configuration: 60% L1 attributes + 40% embeddings")


Checkpoint keys: ['model_state', 'config', 'epoch', 'attribute_names', 'attribute_weights']
Backbone: convnext_tiny
Embedding dimension: 256
Attributes: 40
Checkpoint loaded successfully.

SUBMISSION PACKAGE BUILT
Folder: /content/drive/MyDrive/convnext_tiny_hybrid/final_submission
ZIP: /content/drive/MyDrive/convnext_tiny_hybrid/final_submission.zip
ZIP size (MB): 206.53
Selected configuration: 60% L1 attributes + 40% embeddings


Audit Actual submission runner

In [41]:

from pathlib import Path
import zipfile

out = Path("/content/drive/MyDrive/convnext_tiny_hybrid/final_submission")
zip_path = out.with_suffix(".zip")

print("=" * 65)
print("1. SUBMISSION RUNNER")
print("=" * 65)

run_path = out / "run.py"
runner_text = run_path.read_text()

print(runner_text[:18000])

print("\n" + "=" * 65)
print("2. PACKAGE CONTENTS")
print("=" * 65)

with zipfile.ZipFile(zip_path) as z:
    names = z.namelist()
    for name in names:
        info = z.getinfo(name)
        print(f"{name}  ({info.file_size / 1024**2:.2f} MB)")

print("\n" + "=" * 65)
print("3. REQUIRED FILE CHECK")
print("=" * 65)

required = [
    "run.py",
    "metadata.yaml",
    "assets/config.json",
    "assets/image_model.ts",
    "assets/query_model.ts",
]

for name in required:
    print(f"{'OK' if (out / name).exists() else 'MISSING'}: {name}")

print("\n" + "=" * 65)
print("4. HYBRID CONFIG")
print("=" * 65)

import json
config = json.loads((out / "assets/config.json").read_text())
print("Mode:", config["mode"])
print("Alpha:", config["alpha"])
print("Attribute weights:", len(config["attribute_weights"]))
print("Embedding dimension expected: 256")


1. SUBMISSION RUNNER

from __future__ import annotations
import json, os, sys
from typing import Any
import numpy as np
import torch

HERE=os.path.dirname(os.path.abspath(__file__))
VENDOR=os.path.join(HERE,"vendor")
if os.path.isdir(VENDOR):
    sys.path.insert(0,VENDOR)
from PIL import Image

ASSETS=os.path.join(HERE,"assets")
MODEL=None; QMODEL=None; CFG=None; DEVICE=None

def load_model():
    global MODEL,QMODEL,CFG,DEVICE
    if MODEL is not None: return
    with open(os.path.join(ASSETS,"config.json"),"r") as f: CFG=json.load(f)
    DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    MODEL=torch.jit.load(os.path.join(ASSETS,"image_model.ts"),map_location=DEVICE).eval()
    qp=os.path.join(ASSETS,"query_model.ts")
    if os.path.exists(qp): QMODEL=torch.jit.load(qp,map_location=DEVICE).eval()

def _image(path):
    with Image.open(path) as im:
        im=im.convert("RGB")
        rs=Image.Resampling.BILINEAR if hasattr(Image,"Resampling") else Image.BILINEAR
 

This cell will print the competition metadata and check the Python environment, find real images and call rank_gallery() on a tiny sample. Inference only test btw

In [42]:

import sys
import json
import importlib.util
from pathlib import Path
import numpy as np
import torch

OUT = Path("/content/drive/MyDrive/convnext_tiny_hybrid/final_submission")
PIPE = Path("/content/upar_pipeline/extracted")

print("Python:", sys.version)
print("PyTorch:", torch.__version__)

print("\n--- COMPETITION METADATA ---")
print((OUT / "metadata.yaml").read_text())

print("\n--- IMAGE DEPENDENCY ---")
try:
    import PIL
    print("Pillow version:", PIL.__version__)
    print("Pillow location:", PIL.__file__)
except Exception as e:
    print("Pillow import failed:", repr(e))

print("\n--- FIND REAL TEST IMAGES ---")
extensions = {".jpg", ".jpeg", ".png", ".webp"}
image_paths = []

for root in [PIPE, Path("/content/UPAR-Challenge-2027")]:
    if root.exists():
        for p in root.rglob("*"):
            if p.is_file() and p.suffix.lower() in extensions:
                image_paths.append(p)
                if len(image_paths) >= 3:
                    break
    if len(image_paths) >= 3:
        break

if len(image_paths) < 2:
    raise RuntimeError(
        f"Found only {len(image_paths)} images. "
        "We need at least two real images for the smoke test."
    )

print("Images found:")
for p in image_paths:
    print(" ", p)

print("\n--- CALL SUBMISSION INTERFACE ---")
spec = importlib.util.spec_from_file_location(
    "submission_run", OUT / "run.py"
)
submission_run = importlib.util.module_from_spec(spec)
spec.loader.exec_module(submission_run)

# Use two cached query attribute vectors and real gallery images.
query_sample = np.asarray(queries[:2], dtype=np.float32)

sample = {
    "attribute_names": list(attribute_names),
    "queries": query_sample,
    "gallery": [
        {"image_path": str(p)} for p in image_paths[:3]
    ],
}

result = submission_run.rank_gallery(sample)
distances = result["distances"]

print("Distance matrix shape:", distances.shape)
print("Distances finite:", bool(np.isfinite(distances).all()))
print("Distances:\n", distances)

assert distances.shape == (2, 3), (
    f"Expected (2, 3), got {distances.shape}"
)
assert np.isfinite(distances).all(), "Non-finite distances detected."

print("\nSMOKE TEST PASSED")


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130

--- COMPETITION METADATA ---
# Organizer-compatible code submission marker. Ingestion imports run.py.
command: python3 -c "print('Submission package ready. The ingestion program imports run.py')"


--- IMAGE DEPENDENCY ---
Pillow version: 11.3.0
Pillow location: /usr/local/lib/python3.13/dist-packages/PIL/__init__.py

--- FIND REAL TEST IMAGES ---
Images found:
  /content/UPAR-Challenge-2027/data/PETA/images/17669.png
  /content/UPAR-Challenge-2027/data/PETA/images/16951.png
  /content/UPAR-Challenge-2027/data/PETA/images/12129.png

--- CALL SUBMISSION INTERFACE ---
Distance matrix shape: (2, 3)
Distances finite: True
Distances:
 [[0.4974079  0.4571398  0.31016314]
 [0.47116828 0.4616366  0.3146326 ]]

SMOKE TEST PASSED


Compatibility check

In [44]:

from pathlib import Path
import subprocess
import sys
import json
import zipfile

OUT = Path("/content/drive/MyDrive/convnext_tiny_hybrid/final_submission")
ZIP_PATH = OUT.with_suffix(".zip")

# Use two queries and three actual gallery images.
image_paths = [
    "/content/UPAR-Challenge-2027/data/PETA/images/17669.png",
    "/content/UPAR-Challenge-2027/data/PETA/images/16951.png",
    "/content/UPAR-Challenge-2027/data/PETA/images/12129.png",
]

payload = {
    "queries": queries[:2].tolist(),
    "attribute_names": list(attribute_names),
    "gallery_paths": image_paths,
}

payload_path = OUT / "smoke_test_input.json"
payload_path.write_text(json.dumps(payload))

test_script = r'''
import os, sys, json, importlib.util
import numpy as np

out = sys.argv[1]
payload_path = sys.argv[2]

spec = importlib.util.spec_from_file_location(
    "submission_run", os.path.join(out, "run.py")
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

import PIL
print("Pillow loaded by runner:", PIL.__version__)
print("Pillow path:", PIL.__file__)

with open(payload_path) as f:
    p = json.load(f)

sample = {
    "attribute_names": p["attribute_names"],
    "queries": np.asarray(p["queries"], dtype=np.float32),
    "gallery": [
        {"image_path": x} for x in p["gallery_paths"]
    ],
}

result = mod.rank_gallery(sample)
d = result["distances"]

print("Distance shape:", d.shape)
print("Finite:", bool(np.isfinite(d).all()))
print("Minimum / maximum:", float(d.min()), float(d.max()))

assert d.shape == (2, 3)
assert np.isfinite(d).all()
print("FRESH-PROCESS TEST PASSED")
'''

proc = subprocess.run(
    [sys.executable, "-c", test_script, str(OUT), str(payload_path)],
    capture_output=True,
    text=True,
    timeout=300,
)

print(proc.stdout)
if proc.stderr:
    print("STDERR:\n", proc.stderr[-5000:])

assert proc.returncode == 0, (
    "Fresh-process test failed. Fix this before submission."
)

print("\n--- ZIP INTEGRITY ---")
with zipfile.ZipFile(ZIP_PATH) as z:
    bad = z.testzip()
    print("ZIP integrity:", "OK" if bad is None else f"FAILED: {bad}")
    print("Files in ZIP:", len(z.namelist()))
    print("ZIP size (MB):", round(ZIP_PATH.stat().st_size / 1024**2, 2))

payload_path.unlink(missing_ok=True)


Pillow loaded by runner: 10.4.0
Pillow path: /content/drive/MyDrive/convnext_tiny_hybrid/final_submission/vendor/PIL/__init__.py
Distance shape: (2, 3)
Finite: True
Minimum / maximum: 0.31016314029693604 0.4974079132080078
FRESH-PROCESS TEST PASSED


--- ZIP INTEGRITY ---
ZIP integrity: OK
Files in ZIP: 235
ZIP size (MB): 206.53
